# 06 - Training the surrogate model

Goal: train a LightGBM model that predicts the noise level (dB)
from the **paper's urban morphology features** (notebook 04).

This is our extension of the paper: they show morphology correlates with SPL,
we train a predictive model on it.

**Input** : `data/processed/uganda/sunbird_morphology.parquet` (notebook 04)
**Output** : `outputs/models/surrogate_lgbm.pkl`

This model was then to be calibrated on the Hanoi measurements - a plan later abandoned, see docs/negative-results.md.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib

df = pd.read_parquet('../data/processed/uganda/sunbird_morphology.parquet')
# format='ISO8601': Sunbird timestamps mix with and without microseconds
df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601')
df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
print(f'{len(df)} examples loaded')
df.head()

In [ ]:
# Features = the paper's morphology metrics + time
FEATURES = ['building_density_km2', 'road_density_km_km2', 'intersection_count',
            'dist_road_m', 'hour', 'is_weekend']
TARGET   = 'noise_measurement'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.2f} dB')
print(f'RMSE : {rmse:.2f} dB')
print(f'R²   : {r2:.3f}')

In [ ]:
# Importance des features
lgb.plot_importance(model, figsize=(8, 4), title='Feature importance')
plt.tight_layout()
plt.savefig('../results/figures/sunbird/feature_importance.png', dpi=150)
plt.show()

# Predicted vs actual
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=10)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax.set_xlabel('Measured (dB)')
ax.set_ylabel('Predicted (dB)')
ax.set_title(f'Surrogate model — R²={r2:.3f}')
plt.tight_layout()
plt.savefig('../results/figures/sunbird/pred_vs_real.png', dpi=150)
plt.show()

In [ ]:
# Save the pretrained model
joblib.dump(model, '../models/surrogate_lgbm.pkl')
print('Model saved to models/surrogate_lgbm.pkl')